## Provenance Plots

The cells below query rotating-cylinders benchmark results from ROHub and plot the provenance data directly, instead of reading local result files.


## Setup

Import the shared RoHub query helpers from the `semantic-benchmark` package and the local plotting utilities.

In [ ]:
import sys

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

candidate_paths = [
    Path.cwd() / "provenance",
    Path.cwd().parent / "provenance",
]

for candidate in candidate_paths:
    if candidate.exists():
        provenance_path = candidate.resolve()
        break
else:
    raise FileNotFoundError(
        "Could not find the local provenance helpers directory."
    )

if str(provenance_path) not in sys.path:
    sys.path.append(str(provenance_path))

from semantic_benchmark.rohub import (
    configure_rohub,
    find_annotated_ro_uuids,
    find_named_graphs_for_uuids,
    query_metric_data_from_named_graphs,
)
from plot_metrics import select_plot_columns, plot_provenance_graph


## Configuration

In [ ]:
BENCHMARK_NAME = "rotating-cylinders"
USE_PRODUCTION_ROHUB = True  # set to False to use the development RoHub instance


## Connect to RoHub and find RO-Crates

Configure the RoHub client and look up all RO-Crates that are annotated with this benchmark and the current branch of the code repository.
Each RO-Crate corresponds to one simulation run and contains the recorded parameters and metrics.

In [ ]:
configure_rohub(use_production_rohub=USE_PRODUCTION_ROHUB)

uuids = find_annotated_ro_uuids(
    BENCHMARK_NAME
)
print(f"Found {len(uuids)} RO-Crate(s)")

if not uuids:
    raise RuntimeError(
        "No rotating-cylinders RO-Crates were found in ROHub for this repository URL."
    )


## Resolve SPARQL named graphs

Each RO-Crate is stored in a dedicated named graph in the RoHub triple store.
Resolving the UUIDs to graph URIs allows the subsequent data query to target only the relevant graphs.

In [ ]:
named_graphs = find_named_graphs_for_uuids(
    uuids,
    use_production_rohub=USE_PRODUCTION_ROHUB,
)

print(f"Resolved {len(named_graphs)} named graph(s):")
for uuid, graph_uri in named_graphs.items():
    print(f"  {uuid}\n    -> {graph_uri}")

if not named_graphs:
    raise RuntimeError("No named graphs were resolved for the discovered RO-Crates.")


## Fetch simulation results

Query the named graphs for the simulation parameters and result metrics.
The result is a pandas DataFrame where each row is one simulation run.

In [ ]:
parameters = ["cells_radial"]
metrics = ["l2_error_pressure_rel", "l2_error_velocity_rel"]

data = query_metric_data_from_named_graphs(
    parameters=parameters,
    metrics=metrics,
    named_graphs=list(named_graphs.values()),
)

if data.empty:
    raise RuntimeError("The ROHub query returned no rotating-cylinders metric data.")

data


## Relative L2 Pressure Error

Plot the relative L2 pressure error against the number of radial cells. Lower values indicate closer agreement with the analytical circular Couette-flow reference.

In [ ]:
plot_df = select_plot_columns(
    data,
    parameters=["cells_radial"],
    metrics=["l2_error_pressure_rel"],
)

plot_provenance_graph(
    data=plot_df.values.tolist(),
    x_axis_label="Radial Cells",
    y_axis_label="Relative L2 Pressure Error",
    title="Rotating Cylinders Pressure Error vs Radial Cells",
    log_y=True,
)

## Relative L2 Velocity Error

Plot the relative L2 velocity error against the number of radial cells. The log-log view makes the convergence trend visible as the mesh is refined.

In [ ]:
plot_df = select_plot_columns(
    data,
    parameters=["cells_radial"],
    metrics=["l2_error_velocity_rel"],
)

plot_provenance_graph(
    data=plot_df.values.tolist(),
    x_axis_label="Radial Cells",
    y_axis_label="Relative L2 Velocity Error",
    title="Rotating Cylinders Velocity Error vs Radial Cells",
    log_y=True,
)

## Pressure and Velocity Convergence

Show pressure and velocity errors together for each simulation tool.

In [ ]:
convergence_df = data.loc[:, [
    "tool_name",
    "cells_radial",
    "l2_error_pressure_rel",
    "l2_error_velocity_rel",
]].copy()

for column in ["cells_radial", "l2_error_pressure_rel", "l2_error_velocity_rel"]:
    convergence_df[column] = pd.to_numeric(convergence_df[column], errors="coerce")

convergence_df = convergence_df.dropna().sort_values(["tool_name", "cells_radial"])

fig, ax = plt.subplots(figsize=(10, 6))

for tool_name, tool_df in convergence_df.groupby("tool_name"):
    tool_df = tool_df.sort_values("cells_radial")
    ax.plot(
        tool_df["cells_radial"],
        tool_df["l2_error_pressure_rel"],
        marker="o",
        linewidth=1.5,
        label=f"{tool_name} pressure",
    )
    ax.plot(
        tool_df["cells_radial"],
        tool_df["l2_error_velocity_rel"],
        marker="s",
        linestyle="--",
        linewidth=1.5,
        label=f"{tool_name} velocity",
    )

x_ticks = sorted(convergence_df["cells_radial"].unique())
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Radial Cells")
ax.set_ylabel("Relative L2 Error")
ax.set_title("Rotating Cylinders Pressure and Velocity Convergence")
ax.set_xticks(x_ticks)
ax.set_xticklabels([str(int(x)) if float(x).is_integer() else str(x) for x in x_ticks])
ax.grid(True, which="both", linestyle="-", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()